Metrics visualisation, RF, LGBM, AdaBoost added

# Libraries

In [1]:
import os
from pathlib import Path
import seaborn as sns
import joblib
import matplotlib.pyplot as plt
import pandas as pd

# Data load

In [18]:
df_metrics = pd.read_excel('../results/metrics_modelling3.xlsx')
df_metrics = df_metrics[
    df_metrics['model'].apply(lambda x: "_Stk_diam" in x or '_kl' in x)]

## Splashing data

In [19]:
df_spalshing = df_metrics[df_metrics['target']=='splashing'].drop(38)
df_spalshing = df_spalshing.sort_values(by=['cv_test_f1_median', 'cv_test_roc_auc_median'], ascending=False)
df_spalshing

,dataset,target,model,params,holdout_test_accuracy,holdout_test_precision,holdout_test_recall,holdout_test_f1,holdout_test_roc_auc,holdout_train_accuracy,...,cv_train_precision_median,cv_train_precision_std,cv_train_recall,cv_train_recall_mean,cv_train_recall_median,cv_train_recall_std,cv_train_roc_auc,cv_train_roc_auc_mean,cv_train_roc_auc_median,cv_train_roc_auc_std
44,df_dimless,splashing,CatBoostClassifier_splashing_Stk_diam,{'estimator': <catboost.core.CatBoostClassifie...,0.880000,0.867925,0.958333,0.910891,0.955247,0.996633,...,0.995169,0.001695,"1.0, 1.0, 1.0, 0.9951456310679612, 1.0, 1.0, 1.0",0.999307,1.000000,0.001699,"0.9992877185409023, 0.9999572100984168, 0.9993...",0.999585,0.999549,0.000269
54,df_dimless,splashing,RandomForestClassifier_splashing_Stk_diam,"{'estimator': RandomForestClassifier(), 'sourc...",0.866667,0.851852,0.958333,0.901961,0.954090,0.996633,...,0.995169,0.002185,"1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0",1.000000,1.000000,0.000000,"0.9999784157133607, 0.9999999999999999, 0.9999...",0.999985,0.999979,0.000010
56,df_dimless,splashing,LGBMClassifier_splashing_Stk_diam,"{'estimator': LGBMClassifier(), 'source_featur...",0.880000,0.882353,0.937500,0.909091,0.951775,0.996633,...,0.995169,0.002185,"1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0",1.000000,1.000000,0.000000,"0.9999784157133607, 1.0, 0.9999785204914512, 0...",0.999985,0.999979,0.000010
46,df_dimless,splashing,XGBClassifier_splashing_Stk_diam,"{'estimator': XGBClassifier(base_score=None, b...",0.880000,0.867925,0.958333,0.910891,0.940972,0.996633,...,0.995169,0.002185,"1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0",1.000000,1.000000,0.000000,"0.9999784157133607, 0.9999999999999999, 0.9999...",0.999985,0.999979,0.000010
40,df_dimless,splashing,KNeighborsClassifier_splashing_Stk_diam,"{'estimator': KNeighborsClassifier(), 'source_...",0.853333,0.862745,0.916667,0.888889,0.880787,0.892256,...,0.917874,0.008230,"0.9024390243902439, 0.9317073170731708, 0.9271...",0.916667,0.917476,0.010215,"0.9671918843082237, 0.9732563115104835, 0.9687...",0.968197,0.967738,0.002584
52,df_dimless,splashing,AdaBoostClassifier_splashing_Stk_diam,"{'estimator': AdaBoostClassifier(), 'source_fe...",0.813333,0.840000,0.875000,0.857143,0.881559,0.962963,...,0.941748,0.009907,"0.9463414634146341, 0.9365853658536586, 0.9368...",0.947214,0.946341,0.010240,"0.9849341679257501, 0.9855370132648695, 0.9842...",0.986456,0.985537,0.003393
36,df_dimless,splashing,Logit_splashing_Stk_diam,{'estimator': StatsModelsEstimator(model_class...,0.800000,0.823529,0.875000,0.848485,0.874228,0.845118,...,0.873096,0.009067,"0.8341463414634146, 0.8390243902439024, 0.8349...",0.845821,0.844660,0.010156,"0.8959421541118067, 0.9114676936243047, 0.9020...",0.905980,0.907788,0.006070
42,df_dimless,splashing,SVC_splashing_Stk_diam,"{'estimator': SVC(probability=True), 'source_f...",0.813333,0.840000,0.875000,0.857143,0.911265,0.885522,...,0.911765,0.009098,"0.8975609756097561, 0.8878048780487805, 0.8932...",0.894442,0.893204,0.005054,"0.9373839844593135, 0.9583654257595208, 0.9418...",0.946692,0.946903,0.007162
50,df_dimless,splashing,DecisionStump_splashing_greater_or_equal_kl,{'estimator': DecisionStumpEstimator(less_sign...,0.773333,0.878049,0.750000,0.808989,0.782407,0.774411,...,0.885057,0.008861,"0.7170731707317073, 0.751219512195122, 0.74757...",0.741656,0.747573,0.013057,"0.7700410101446147, 0.7966623876765083, 0.7852...",0.787489,0.785720,0.009057


In [20]:
df_spalshing['model'] = df_spalshing['model'].apply(lambda x: x.split('_')[0])
df_spalshing = df_spalshing.iloc[: ,2:]
df_spalshing.replace({'Logit': 'LogisticRegression', 'DecisionStump': 'Baseline'}, inplace=True)
df_spalshing.rename(columns={'model': 'Model',
                             'cv_test_accuracy': 'Accuracy',
                             'cv_test_f1': 'F1',
                             'cv_test_precision': 'Precision',
                             'cv_test_recall': 'Recall',
                             'cv_test_roc_auc': 'ROC AUC'}, inplace=True)
df_spalshing['Target'] = 'Splashing'

# df_spalshing.drop(columns=['Precision', 'Recall'], inplace=True)
df_spalshing

,Model,params,holdout_test_accuracy,holdout_test_precision,holdout_test_recall,holdout_test_f1,holdout_test_roc_auc,holdout_train_accuracy,holdout_train_precision,holdout_train_recall,...,cv_train_precision_std,cv_train_recall,cv_train_recall_mean,cv_train_recall_median,cv_train_recall_std,cv_train_roc_auc,cv_train_roc_auc_mean,cv_train_roc_auc_median,cv_train_roc_auc_std,Target
44,CatBoostClassifier,{'estimator': <catboost.core.CatBoostClassifie...,0.880000,0.867925,0.958333,0.910891,0.955247,0.996633,0.994819,1.000000,...,0.001695,"1.0, 1.0, 1.0, 0.9951456310679612, 1.0, 1.0, 1.0",0.999307,1.000000,0.001699,"0.9992877185409023, 0.9999572100984168, 0.9993...",0.999585,0.999549,0.000269,Splashing
54,RandomForestClassifier,"{'estimator': RandomForestClassifier(), 'sourc...",0.866667,0.851852,0.958333,0.901961,0.954090,0.996633,0.994819,1.000000,...,0.002185,"1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0",1.000000,1.000000,0.000000,"0.9999784157133607, 0.9999999999999999, 0.9999...",0.999985,0.999979,0.000010,Splashing
56,LGBMClassifier,"{'estimator': LGBMClassifier(), 'source_featur...",0.880000,0.882353,0.937500,0.909091,0.951775,0.996633,0.994819,1.000000,...,0.002185,"1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0",1.000000,1.000000,0.000000,"0.9999784157133607, 1.0, 0.9999785204914512, 0...",0.999985,0.999979,0.000010,Splashing
46,XGBClassifier,"{'estimator': XGBClassifier(base_score=None, b...",0.880000,0.867925,0.958333,0.910891,0.940972,0.996633,0.994819,1.000000,...,0.002185,"1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0",1.000000,1.000000,0.000000,"0.9999784157133607, 0.9999999999999999, 0.9999...",0.999985,0.999979,0.000010,Splashing
40,KNeighborsClassifier,"{'estimator': KNeighborsClassifier(), 'source_...",0.853333,0.862745,0.916667,0.888889,0.880787,0.892256,0.916667,0.916667,...,0.008230,"0.9024390243902439, 0.9317073170731708, 0.9271...",0.916667,0.917476,0.010215,"0.9671918843082237, 0.9732563115104835, 0.9687...",0.968197,0.967738,0.002584,Splashing
52,AdaBoostClassifier,"{'estimator': AdaBoostClassifier(), 'source_fe...",0.813333,0.840000,0.875000,0.857143,0.881559,0.962963,0.959391,0.984375,...,0.009907,"0.9463414634146341, 0.9365853658536586, 0.9368...",0.947214,0.946341,0.010240,"0.9849341679257501, 0.9855370132648695, 0.9842...",0.986456,0.985537,0.003393,Splashing
36,LogisticRegression,{'estimator': StatsModelsEstimator(model_class...,0.800000,0.823529,0.875000,0.848485,0.874228,0.845118,0.901099,0.854167,...,0.009067,"0.8341463414634146, 0.8390243902439024, 0.8349...",0.845821,0.844660,0.010156,"0.8959421541118067, 0.9114676936243047, 0.9020...",0.905980,0.907788,0.006070,Splashing
42,SVC,"{'estimator': SVC(probability=True), 'source_f...",0.813333,0.840000,0.875000,0.857143,0.911265,0.885522,0.938889,0.880208,...,0.009098,"0.8975609756097561, 0.8878048780487805, 0.8932...",0.894442,0.893204,0.005054,"0.9373839844593135, 0.9583654257595208, 0.9418...",0.946692,0.946903,0.007162,Splashing
50,Baseline,{'estimator': DecisionStumpEstimator(less_sign...,0.773333,0.878049,0.750000,0.808989,0.782407,0.774411,0.893082,0.739583,...,0.008861,"0.7170731707317073, 0.751219512195122, 0.74757...",0.741656,0.747573,0.013057,"0.7700410101446147, 0.7966623876765083, 0.7852...",0.787489,0.785720,0.009057,Splashing


## No fragmentation data

In [21]:
df_no_fragmentation = df_metrics[df_metrics['target']=='no_fragmentation'].drop(39)
df_no_fragmentation = df_no_fragmentation.sort_values(by=['cv_test_f1_median', 'cv_test_roc_auc_median'], ascending=False)
df_no_fragmentation

,dataset,target,model,params,holdout_test_accuracy,holdout_test_precision,holdout_test_recall,holdout_test_f1,holdout_test_roc_auc,holdout_train_accuracy,...,cv_train_precision_median,cv_train_precision_std,cv_train_recall,cv_train_recall_mean,cv_train_recall_median,cv_train_recall_std,cv_train_roc_auc,cv_train_roc_auc_mean,cv_train_roc_auc_median,cv_train_roc_auc_std
45,df_dimless,no_fragmentation,CatBoostClassifier_no_fragmentation_Stk_diam,{'estimator': <catboost.core.CatBoostClassifie...,0.920000,0.850000,0.85,0.850000,0.986364,0.996633,...,0.988372,0.005823,"0.9880952380952381, 1.0, 0.9882352941176471, 0...",0.991577,0.988235,0.005328,"0.999949124949125, 1.0, 0.9999497234791352, 1....",0.999957,0.999950,4.189518e-05
57,df_dimless,no_fragmentation,LGBMClassifier_no_fragmentation_Stk_diam,"{'estimator': LGBMClassifier(), 'source_featur...",0.933333,0.857143,0.90,0.878049,0.985455,1.000000,...,1.000000,0.000000,"1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0",1.000000,1.000000,0.000000,"1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0",1.000000,1.000000,0.000000e+00
47,df_dimless,no_fragmentation,XGBClassifier_no_fragmentation_Stk_diam,"{'estimator': XGBClassifier(base_score=None, b...",0.960000,1.000000,0.85,0.918919,0.991818,1.000000,...,1.000000,0.000000,"1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0",1.000000,1.000000,0.000000,"0.9999999999999999, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0",1.000000,1.000000,4.196249e-17
55,df_dimless,no_fragmentation,RandomForestClassifier_no_fragmentation_Stk_diam,"{'estimator': RandomForestClassifier(), 'sourc...",0.946667,0.900000,0.90,0.900000,0.983636,1.000000,...,1.000000,0.000000,"1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0",1.000000,1.000000,0.000000,"1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0",1.000000,1.000000,0.000000e+00
41,df_dimless,no_fragmentation,KNeighborsClassifier_no_fragmentation_Stk_diam,"{'estimator': KNeighborsClassifier(), 'source_...",0.880000,0.789474,0.75,0.769231,0.947727,0.912458,...,0.808989,0.008902,"0.8452380952380952, 0.8941176470588236, 0.8823...",0.853521,0.858824,0.031415,"0.9700091575091574, 0.9787078934137757, 0.9681...",0.970351,0.969583,3.602719e-03
53,df_dimless,no_fragmentation,AdaBoostClassifier_no_fragmentation_Stk_diam,"{'estimator': AdaBoostClassifier(), 'source_fe...",0.866667,0.727273,0.80,0.761905,0.894545,0.976431,...,0.954023,0.013843,"0.9642857142857143, 0.9882352941176471, 0.9529...",0.964646,0.964706,0.014062,"0.9983465608465608, 0.998340874811463, 0.99592...",0.997347,0.997436,9.362422e-04
37,df_dimless,no_fragmentation,Logit_no_fragmentation_Stk_diam,{'estimator': StatsModelsEstimator(model_class...,0.906667,0.809524,0.85,0.829268,0.951818,0.861953,...,0.767442,0.027567,"0.7857142857142857, 0.788235294117647, 0.74117...",0.760984,0.752941,0.021087,"0.9395604395604396, 0.9486676721970839, 0.9361...",0.940673,0.941277,5.872686e-03
43,df_dimless,no_fragmentation,SVC_no_fragmentation_Stk_diam,"{'estimator': SVC(probability=True), 'source_f...",0.866667,0.750000,0.75,0.750000,0.954545,0.895623,...,0.795181,0.016414,"0.7261904761904762, 0.8235294117647058, 0.7529...",0.750800,0.752941,0.042170,"0.9537545787545788, 0.9613876319758673, 0.9489...",0.954066,0.951986,4.703518e-03
51,df_dimless,no_fragmentation,DecisionStump_no_fragmentation_less_kl,"{'estimator': DecisionStumpEstimator(), 'sourc...",0.733333,0.500000,0.90,0.642857,0.786364,0.740741,...,0.500000,0.010062,"0.8928571428571429, 0.8823529411764706, 0.8705...",0.878812,0.882353,0.011901,"0.7861721611721613, 0.7916038210155857, 0.7878...",0.783728,0.786172,6.631068e-03


In [22]:
df_no_fragmentation = df_no_fragmentation.iloc[: ,2:]
df_no_fragmentation['model'] = df_no_fragmentation['model'].apply(lambda x: x.split('_')[0])
df_no_fragmentation.replace({'Logit': 'LogisticRegression', 'DecisionStump': 'Baseline'}, inplace=True)
df_no_fragmentation.rename(columns={'model': 'Model',
                             'cv_test_accuracy': 'Accuracy',
                             'cv_test_f1': 'F1',
                             'cv_test_precision': 'Precision',
                             'cv_test_recall': 'Recall',
                             'cv_test_roc_auc': 'ROC AUC'}, inplace=True)
df_no_fragmentation['Target'] = 'No fragmentation'
# df_no_fragmentation.drop(columns=['Precision', 'Recall'], inplace=True)
df_no_fragmentation

,Model,params,holdout_test_accuracy,holdout_test_precision,holdout_test_recall,holdout_test_f1,holdout_test_roc_auc,holdout_train_accuracy,holdout_train_precision,holdout_train_recall,...,cv_train_precision_std,cv_train_recall,cv_train_recall_mean,cv_train_recall_median,cv_train_recall_std,cv_train_roc_auc,cv_train_roc_auc_mean,cv_train_roc_auc_median,cv_train_roc_auc_std,Target
45,CatBoostClassifier,{'estimator': <catboost.core.CatBoostClassifie...,0.920000,0.850000,0.85,0.850000,0.986364,0.996633,0.987500,1.000000,...,0.005823,"0.9880952380952381, 1.0, 0.9882352941176471, 0...",0.991577,0.988235,0.005328,"0.999949124949125, 1.0, 0.9999497234791352, 1....",0.999957,0.999950,4.189518e-05,No fragmentation
57,LGBMClassifier,"{'estimator': LGBMClassifier(), 'source_featur...",0.933333,0.857143,0.90,0.878049,0.985455,1.000000,1.000000,1.000000,...,0.000000,"1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0",1.000000,1.000000,0.000000,"1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0",1.000000,1.000000,0.000000e+00,No fragmentation
47,XGBClassifier,"{'estimator': XGBClassifier(base_score=None, b...",0.960000,1.000000,0.85,0.918919,0.991818,1.000000,1.000000,1.000000,...,0.000000,"1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0",1.000000,1.000000,0.000000,"0.9999999999999999, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0",1.000000,1.000000,4.196249e-17,No fragmentation
55,RandomForestClassifier,"{'estimator': RandomForestClassifier(), 'sourc...",0.946667,0.900000,0.90,0.900000,0.983636,1.000000,1.000000,1.000000,...,0.000000,"1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0",1.000000,1.000000,0.000000,"1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0",1.000000,1.000000,0.000000e+00,No fragmentation
41,KNeighborsClassifier,"{'estimator': KNeighborsClassifier(), 'source_...",0.880000,0.789474,0.75,0.769231,0.947727,0.912458,0.835443,0.835443,...,0.008902,"0.8452380952380952, 0.8941176470588236, 0.8823...",0.853521,0.858824,0.031415,"0.9700091575091574, 0.9787078934137757, 0.9681...",0.970351,0.969583,3.602719e-03,No fragmentation
53,AdaBoostClassifier,"{'estimator': AdaBoostClassifier(), 'source_fe...",0.866667,0.727273,0.80,0.761905,0.894545,0.976431,0.961538,0.949367,...,0.013843,"0.9642857142857143, 0.9882352941176471, 0.9529...",0.964646,0.964706,0.014062,"0.9983465608465608, 0.998340874811463, 0.99592...",0.997347,0.997436,9.362422e-04,No fragmentation
37,LogisticRegression,{'estimator': StatsModelsEstimator(model_class...,0.906667,0.809524,0.85,0.829268,0.951818,0.861953,0.737500,0.746835,...,0.027567,"0.7857142857142857, 0.788235294117647, 0.74117...",0.760984,0.752941,0.021087,"0.9395604395604396, 0.9486676721970839, 0.9361...",0.940673,0.941277,5.872686e-03,No fragmentation
43,SVC,"{'estimator': SVC(probability=True), 'source_f...",0.866667,0.750000,0.75,0.750000,0.954545,0.895623,0.807692,0.797468,...,0.016414,"0.7261904761904762, 0.8235294117647058, 0.7529...",0.750800,0.752941,0.042170,"0.9537545787545788, 0.9613876319758673, 0.9489...",0.954066,0.951986,4.703518e-03,No fragmentation
51,Baseline,"{'estimator': DecisionStumpEstimator(), 'sourc...",0.733333,0.500000,0.90,0.642857,0.786364,0.740741,0.507353,0.873418,...,0.010062,"0.8928571428571429, 0.8823529411764706, 0.8705...",0.878812,0.882353,0.011901,"0.7861721611721613, 0.7916038210155857, 0.7878...",0.783728,0.786172,6.631068e-03,No fragmentation


# Data plot

In [23]:
path_visualisations = Path('../results/metrics_visualisation3/')
if not os.path.exists(path_visualisations): os.makedirs(path_visualisations)

# Combined plots

In [60]:
columns = ['Model', 'Accuracy', 'F1', 'Precision', 'Recall', 'ROC AUC']
# columns = ['Model', 'Accuracy', 'F1', 'ROC AUC']

df_spalshing_plot = df_spalshing[columns].copy()
df_no_fragmentation_plot = df_no_fragmentation[columns].copy()

for column in columns[1:]:
    df_spalshing_plot[column] = df_spalshing_plot[column].apply(lambda x: [float(i) for i in x.split(', ')])
    df_no_fragmentation_plot[column] = df_no_fragmentation_plot[column].apply(lambda x: [float(i) for i in x.split(', ')])

df_spalshing_melt = pd.melt(df_spalshing_plot, id_vars=['Model'], value_vars=columns[1:])
df_spalshing_exploded = df_spalshing_melt.explode('value')

df_no_fragmentation_melt = pd.melt(df_no_fragmentation_plot, id_vars=['Model'], value_vars=columns[1:])
df_no_fragmentation_exploded = df_no_fragmentation_melt.explode('value')

In [81]:
sns.set_style('whitegrid', {'font.family':'serif', 'font.serif':'Times New Roman', 'font.size': 28})
plt.rcParams['figure.dpi'] = 200
plt.rcParams["font.family"] = "Times New Roman"
palette = sns.color_palette("husl", len(df_spalshing['Model'].unique()))
fig, axes = plt.subplots(1, 2, figsize=(24, 12), dpi=1200)
plt.rcParams.update({'font.size': 24})

# First plot
sns.barplot(x='variable', y='value', hue='Model', data=df_spalshing_exploded, palette=palette, ax=axes[0], capsize=.02)
axes[0].set_title('(a)')
axes[0].set_ylim(0.4, axes[0].get_ylim()[1])
axes[0].set_xlabel('')
axes[0].set_ylabel('')

# Second plot
sns.barplot(x='variable', y='value', hue='Model', data=df_no_fragmentation_exploded, palette=palette, ax=axes[1], capsize=.02)
axes[1].set_title('(b)')
axes[1].set_xlabel('')
axes[1].set_ylabel('')
axes[1].set_ylim(0.4, axes[0].get_ylim()[1])

# Get legend handles and labels
handles, labels = axes[0].get_legend_handles_labels()

# Remove legends from individual subplots
axes[0].get_legend().remove()
axes[1].get_legend().remove()

# Calculate number of columns needed (half the items per row)
n_cols = (len(labels) + 1) // 2  # This will create 2 rows

# Add figure legend with two rows
fig.legend(handles, labels, title='Model', 
           loc='lower center', 
           ncol=n_cols,  # This creates multiple columns which effectively makes rows
           bbox_to_anchor=(0.5, -0.1))  # Adjust this value if needed for proper positioning

plt.savefig(path_visualisations / f'combined_metrics3_filtered_ieee_added.pdf', bbox_inches='tight')
plt.show()

In [90]:
print('Метрики для splashing')
df_spalshing[['Model', 'cv_test_accuracy_mean', 'cv_test_f1_mean', 'cv_test_roc_auc_mean']].sort_values(by=[
    'cv_test_f1_mean', 'cv_test_roc_auc_mean'], ascending=False)

Метрики для splashing


,Model,cv_test_accuracy_mean,cv_test_f1_mean,cv_test_roc_auc_mean
44,CatBoostClassifier,0.894979,0.918864,0.961575
56,LGBMClassifier,0.894979,0.918850,0.952416
54,RandomForestClassifier,0.884247,0.909723,0.951641
46,XGBClassifier,0.881551,0.906665,0.950007
52,AdaBoostClassifier,0.852052,0.885840,0.900856
40,KNeighborsClassifier,0.846661,0.880597,0.913421
42,SVC,0.833134,0.870091,0.916740
36,LogisticRegression,0.814216,0.853493,0.893077
50,Baseline,0.773884,0.807236,0.787106


In [91]:
print('Метрики для no fragmentation')
df_no_fragmentation[['Model', 'cv_test_accuracy_mean', 'cv_test_f1_mean', 'cv_test_roc_auc_mean']].sort_values(by=[
    'cv_test_f1_mean', 'cv_test_roc_auc_mean'], ascending=False)

Метрики для no fragmentation


,Model,cv_test_accuracy_mean,cv_test_f1_mean,cv_test_roc_auc_mean
45,CatBoostClassifier,0.911301,0.833727,0.957038
57,LGBMClassifier,0.911451,0.824872,0.956724
47,XGBClassifier,0.900569,0.806544,0.959916
55,RandomForestClassifier,0.881851,0.772924,0.941619
53,AdaBoostClassifier,0.865628,0.748960,0.916614
37,LogisticRegression,0.854847,0.732916,0.935706
41,KNeighborsClassifier,0.852101,0.722299,0.917059
43,SVC,0.857492,0.718889,0.937555
51,Baseline,0.739193,0.644887,0.784118


# Percent change with baseline

In [63]:
df_comp_spl = df_spalshing.loc[[50, 36, 44, 56]][['Model', 'cv_test_accuracy_median', 'cv_test_f1_median', 'cv_test_roc_auc_median']]
df_comp_spl['pct_change_cv_test_accuracy'] = (df_comp_spl['cv_test_accuracy_median'].apply(
    lambda x: (x-df_comp_spl['cv_test_accuracy_median'].iloc[0]))*100)
df_comp_spl['pct_change_cv_test_f1'] = (df_comp_spl['cv_test_f1_median'].apply(
    lambda x: (x-df_comp_spl['cv_test_f1_median'].iloc[0]))*100)
df_comp_spl['pct_change_cv_test_roc_auc'] = (df_comp_spl['cv_test_roc_auc_median'].apply(
    lambda x: (x-df_comp_spl['cv_test_roc_auc_median'].iloc[0]))*100)
print('Метрики для splashing')
df_comp_spl.sort_values(by=['pct_change_cv_test_f1', 'pct_change_cv_test_roc_auc'])

Метрики для splashing


,Model,cv_test_accuracy_median,cv_test_f1_median,cv_test_roc_auc_median,pct_change_cv_test_accuracy,pct_change_cv_test_f1,pct_change_cv_test_roc_auc
50,Baseline,0.773585,0.800000,0.797214,0.000000,0.000000,0.000000
36,LogisticRegression,0.830189,0.865672,0.869969,5.660377,6.567164,7.275542
56,LGBMClassifier,0.886792,0.914286,0.945820,11.320755,11.428571,14.860681
44,CatBoostClassifier,0.905660,0.927536,0.953560,13.207547,12.753623,15.634675


In [94]:
df_spalshing[['Model', 'cv_test_accuracy_median', 
              'cv_test_f1_median', 'cv_test_roc_auc_median']].sort_values(by=['cv_test_f1_median', 'cv_test_roc_auc_median'],
                                                                          ascending=False)

,Model,cv_test_accuracy_median,cv_test_f1_median,cv_test_roc_auc_median
44,CatBoostClassifier,0.905660,0.927536,0.953560
54,RandomForestClassifier,0.905660,0.927536,0.939628
56,LGBMClassifier,0.886792,0.914286,0.945820
46,XGBClassifier,0.886792,0.911765,0.942724
40,KNeighborsClassifier,0.830189,0.869565,0.923308
52,AdaBoostClassifier,0.830189,0.865672,0.900929
36,LogisticRegression,0.830189,0.865672,0.869969
42,SVC,0.830189,0.861538,0.928793
50,Baseline,0.773585,0.800000,0.797214


In [64]:
df_comp_no_frgm = df_no_fragmentation.loc[[51, 37, 45, 57]][['Model', 'cv_test_accuracy_median', 'cv_test_f1_median', 'cv_test_roc_auc_median']]
df_comp_no_frgm['pct_change_cv_test_accuracy'] = (df_comp_no_frgm['cv_test_accuracy_median'].apply(
    lambda x: (x-df_comp_no_frgm[df_comp_no_frgm['Model']=='Baseline']['cv_test_accuracy_median']))*100)
df_comp_no_frgm['pct_change_cv_test_f1'] = (df_comp_no_frgm['cv_test_f1_median'].apply(
    lambda x: (x-df_comp_no_frgm[df_comp_no_frgm['Model']=='Baseline']['cv_test_f1_median']))*100)
df_comp_no_frgm['pct_change_cv_test_roc_auc'] = (df_comp_no_frgm['cv_test_roc_auc_median'].apply(
    lambda x: (x-df_comp_no_frgm[df_comp_no_frgm['Model']=='Baseline']['cv_test_roc_auc_median']))*100)
print('Метрики для no fragmentation')
df_comp_no_frgm.sort_values(by=['pct_change_cv_test_f1', 'pct_change_cv_test_roc_auc'])

Метрики для no fragmentation


,Model,cv_test_accuracy_median,cv_test_f1_median,cv_test_roc_auc_median,pct_change_cv_test_accuracy,pct_change_cv_test_f1,pct_change_cv_test_roc_auc
51,Baseline,0.759259,0.647059,0.771795,0.000000,0.000000,0.000000
37,LogisticRegression,0.851852,0.733333,0.932234,9.259259,8.627451,16.043956
57,LGBMClassifier,0.924528,0.846154,0.958974,16.526904,19.909502,18.717949
45,CatBoostClassifier,0.924528,0.846154,0.979487,16.526904,19.909502,20.769231


# Metrics per class

In [ ]:
path_models = Path('results/models_modelling3')